# G2 — the four adapter rows

`verimem-g1rerun-g2` completed stage 1 and the two prompted rows, then died on the first
row that loads an adapter:

```
ImportError: Found an incompatible version of torchao.
Found version 0.10.0, but only versions above 0.16.0 are supported
```

peft raises that from inside `dispatch_torchao` while building the LoRA modules. The SFT
runs never hit it because they install Unsloth, which pins a compatible stack; this
notebook installs peft without it. **torchao is not used here at all** — the 4-bit base
comes from bitsandbytes — so the fix is to get it out of the way rather than to chase a
version that matches.

The rows already finished (`g1-rerun`, `g2-prompted-3b`, `g2-prompted-3b-retrieval`) are
not repeated. Their numbers were recovered from the failed kernel's log, which is the
reason the runners print metrics to stdout.

**An adapter is loaded and generated from before any long stage starts.** The previous
attempt spent an hour on stage 1 and then discovered a broken import; two minutes of
checking first would have saved all of it.

In [ ]:
import os
import pathlib
import subprocess
import sys

os.chdir("/kaggle/working")
REPO = pathlib.Path("/kaggle/working/VeriMem")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/TERABYTE05/VeriMem.git", str(REPO)],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO))

RESULTS = "/kaggle/working/results"
os.environ["VERIMEM_DATA_DIR"] = "/kaggle/working/data"
os.environ["VERIMEM_CACHE_DIR"] = "/kaggle/working/cache"
os.environ["VERIMEM_RESULTS_DIR"] = RESULTS
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

from core.prompts import prompt_fingerprint

print("verifier prompt:", prompt_fingerprint())

## Inputs and the warm cache

In [ ]:
import pathlib
import shutil

DATA = pathlib.Path("/kaggle/working/data/averitec")
TRAJ = pathlib.Path("/kaggle/working/data/trajectories")
for d in (DATA, TRAJ):
    d.mkdir(parents=True, exist_ok=True)
print("attached:", [p.name for p in pathlib.Path("/kaggle/input").iterdir()])

for name, dest in (("dev.json", DATA), ("pool_dev.jsonl", DATA), ("trajectories.jsonl", TRAJ)):
    if (dest / name).exists():
        continue
    for found in pathlib.Path("/kaggle/input").rglob(name):
        shutil.copy(found, dest / name)
        break
    else:
        raise SystemExit(f"{name} not found among the attached inputs")

restored = 0
for ns in ("llm", "evaluator"):
    dst = pathlib.Path(f"/kaggle/working/cache/{ns}")
    if dst.exists():
        continue
    for src in pathlib.Path("/kaggle/input").rglob(f"cache/{ns}"):
        if src.is_dir():
            shutil.copytree(src, dst)
            n = len(list(dst.rglob("*.json")))
            restored += n
            print(f"restored {n} cached {ns} calls")
            break
print(f"{restored} cached calls restored" if restored else "WARNING: cold cache")

## Dependencies, with torchao removed

peft's torchao dispatcher raises on an old version instead of declining, so an unused
package breaks adapter loading. Upgrading is tried first; if no compatible wheel exists
for the image's torch, it is uninstalled, which makes peft's availability check return
False cleanly.

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes openai pyyaml python-dotenv 2>&1 | tail -2

import importlib.metadata as md_


def torchao_version():
    try:
        return md_.version("torchao")
    except md_.PackageNotFoundError:
        return None

print("torchao before:", torchao_version())
if torchao_version() is not None:
    !pip install -q -U "torchao>=0.16.0" 2>&1 | tail -2
    v = torchao_version()
    ok = v is not None and tuple(int(x) for x in v.split(".")[:2]) >= (0, 16)
    if not ok:
        print(f"no compatible wheel (got {v}); removing it - nothing here uses it")
        !pip uninstall -q -y torchao 2>&1 | tail -1
print("torchao after:", torchao_version())

import torch

print("cuda:", torch.cuda.is_available(), "| bf16:", torch.cuda.is_bf16_supported())

## The adapters

In [ ]:
ADAPTERS = {}
for root in (pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")):
    if not root.exists():
        continue
    for cfg in root.rglob("adapter/adapter_config.json"):
        run = cfg.parent.parent.name
        ADAPTERS.setdefault(run.split("_", 1)[-1] if "_" in run else run, cfg.parent)

for k in sorted(ADAPTERS):
    size = sum(f.stat().st_size for f in ADAPTERS[k].rglob("*") if f.is_file())
    print(f"  {k:<18} {size/1e6:7.1f} MB  {ADAPTERS[k]}")

NEEDED = ("lora-sft-gold", "rag-sft-gold")
missing = [k for k in NEEDED if k not in ADAPTERS]
if missing:
    raise SystemExit(f"missing adapters {missing} - attach the verimem-sft-gold output")
OPTIONAL = [k for k in ("lora-sft", "rag-sft") if k in ADAPTERS]
print(f"\nrequired: {list(NEEDED)}  |  filtered comparison: {OPTIONAL or 'not attached'}")

## Preflight: load one adapter and generate

This is the check whose absence cost the previous attempt an hour. It loads an adapter
through the exact code path the rows use and generates a few tokens.

In [ ]:
from core.prompts import VERIFIER_SYSTEM
from training.verifier import FinetunedVerifier

probe = FinetunedVerifier(ADAPTERS["lora-sft-gold"])
reply = probe.complete(
    "Claim: The Kochi metro opened to the public in 2017.\n\nEvidence:\n"
    "[reuters.com] The Kochi Metro began commercial operations in June 2017.",
    system=VERIFIER_SYSTEM,
    max_tokens=48,
)
print("model:", probe.model)
print("reply:", repr(reply.text[:160]))

from agent.baselines import parse_verdict

parsed = parse_verdict(reply.text)
print("parsed verdict:", parsed.verdict, "| parsed ok:", parsed.parsed)
assert parsed.parsed, "the adapter generated something the pipeline cannot read"

# Free it before the rows start, so the first row is not competing for VRAM.
del probe, reply
import gc

import torch

gc.collect()
torch.cuda.empty_cache()
print("\npreflight passed")

## Teacher endpoint

In [ ]:
import shutil
import subprocess
import time
import urllib.request

os.environ["OLLAMA_NUM_PARALLEL"] = "4"   # the student shares this GPU
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

if shutil.which("zstd") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd",
                   shell=True, capture_output=True)
if shutil.which("ollama") is None:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, capture_output=True, text=True)
OLLAMA_BIN = shutil.which("ollama")
assert OLLAMA_BIN, "ollama did not install"
try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
except Exception:
    subprocess.Popen([OLLAMA_BIN, "serve"], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(120):
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not start")
subprocess.run([OLLAMA_BIN, "pull", "qwen2.5:7b"], check=True)
print("teacher up", flush=True)

os.environ["VERIMEM_API_BASE_URL"] = "http://localhost:11434/v1"
os.environ["VERIMEM_API_MODEL"] = "qwen2.5:7b"
os.environ.setdefault("VERIMEM_API_KEY", "ollama")

EVALUATOR = None
for root in (pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")):
    for p in (root.rglob("model/config.json") if root.exists() else []):
        if (p.parent / "model.safetensors").exists() or (p.parent / "pytorch_model.bin").exists():
            EVALUATOR = p.parent
            break
    if EVALUATOR:
        break
assert EVALUATOR, "no evaluator attached"
print("evaluator:", EVALUATOR)

DEV = "/kaggle/working/data/averitec/dev.json"
POOL = "/kaggle/working/data/averitec/pool_dev.jsonl"
TRAJECTORIES = "/kaggle/working/data/trajectories/trajectories.jsonl"

## The four adapter rows

The teacher calls these need were cached by the previous kernel's stage 1 and restored
above, so this is almost entirely student generation. No `--workers`: the student is on
this GPU.

In [ ]:
ROWS = [
    ("g2-lora-sft", "lora-sft-gold", False),
    ("g2-rag-sft",  "rag-sft-gold",  True),
]
for key in OPTIONAL:
    ROWS.append((f"g2-{key}-filtered", key, key.startswith("rag")))

LIMIT = "300"
for name, key, retrieval in ROWS:
    argv = [sys.executable, "-u", "-m", "eval.run_baselines",
            "--data", DEV, "--store", POOL, "--evaluator", str(EVALUATOR),
            "--system", "crag", "--limit", LIMIT, "--yes", "--name", name,
            "--verifier-adapter", str(ADAPTERS[key])]
    if retrieval:
        argv += ["--trajectories", TRAJECTORIES, "--experience-retrieval"]
    print(f"\n######## {name}  adapter={key}  retrieval={retrieval} ########", flush=True)
    r = subprocess.run(argv, env=os.environ)
    if r.returncode != 0:
        raise SystemExit(f"{name} failed")

## The rows this kernel produced

In [ ]:
import json
import pathlib

LABELS = ["Supported", "Refuted", "Not enough evidence", "Conflicting evidence"]
print(f"{'row':<26}{'macro-F1':>10}{'accuracy':>10}" + "".join(f"{label[:12]:>14}" for label in LABELS))
print("-" * (46 + 14 * len(LABELS)))
for f in sorted(pathlib.Path(RESULTS).rglob("metrics.json")):
    name = f.parent.name.split("_", 1)[-1]
    if not name.startswith("g2-"):
        continue
    m = json.loads(f.read_text()).get("crag", {})
    per = m.get("per_class", {})
    line = f"{name:<26}{m.get('macro_f1', 0):>10.3f}{m.get('accuracy', 0):>10.3f}"
    for label in LABELS:
        f1 = per.get(label, {}).get("f1")
        line += f"{'-' if f1 is None else format(f1, '.3f'):>14}"
    print(line)

print("\nThe two prompted rows came from the previous kernel:")
print("  g2-prompted-3b            0.243")
print("  g2-prompted-3b-retrieval  0.285")

In [ ]:
import shutil

shutil.rmtree("/kaggle/working/VeriMem/.git", ignore_errors=True)
for p in pathlib.Path(RESULTS).rglob("checkpoints"):
    shutil.rmtree(p, ignore_errors=True)
!du -sh /kaggle/working/cache /kaggle/working/results